# OSS Maintainer Toolkit — OpenClaw Demo

This notebook demonstrates the **Python API** of the OSS Maintainer Toolkit against
[nicoseng/OpenClaw](https://github.com/nicoseng/OpenClaw), a real open-source project.

The toolkit provides automated PR/issue triage using a three-tier gated pipeline:
1. **Tier 1 — Embeddings:** semantic dedup, linking, conflict detection (free, local)
2. **Tier 2 — Heuristics:** rule-based suspicion scoring (free, deterministic)
3. **Tier 3 — LLM:** vision alignment via OpenRouter free models (optional, disabled here)

**All cells run with Tier 1+2 only — no LLM API key needed.** Set `AUDITOR_GK_GITHUB_TOKEN`
for higher rate limits, or leave unset for public repo access.

```bash
pip install oss-maintainer-toolkit[gatekeeper]
```

In [ ]:
import os

from oss_maintainer_toolkit.gatekeeper.github_client import GitHubClient
from oss_maintainer_toolkit.gatekeeper.ingest import ingest_pr, ingest_batch
from oss_maintainer_toolkit.gatekeeper.issue_ingest import ingest_issue, ingest_issue_batch
from oss_maintainer_toolkit.gatekeeper.dedup import compute_embedding
from oss_maintainer_toolkit.gatekeeper.issue_dedup import compute_issue_embedding
from oss_maintainer_toolkit.gatekeeper.pipeline import run_pipeline
from oss_maintainer_toolkit.gatekeeper.linking import find_issue_pr_links
from oss_maintainer_toolkit.gatekeeper.conflict_detection import detect_conflicts
from oss_maintainer_toolkit.gatekeeper.contributor_profiles import build_contributor_profile
from oss_maintainer_toolkit.gatekeeper.audit_backlog import run_audit
from oss_maintainer_toolkit.gatekeeper.vision import load_vision_document

OWNER = "nicoseng"
REPO = "OpenClaw"

# Reuse a single client across cells
client = GitHubClient()
await client.__aenter__()

rate = await client.check_rate_limit()
print(f"GitHub API: {rate.get('rate', {}).get('remaining', '?')} requests remaining")

## 1. Single PR Assessment

Ingest one PR and run the full Tier 1+2 pipeline. The scorecard shows:
- **Verdict**: `FAST_TRACK`, `REVIEW_REQUIRED`, or `RECOMMEND_CLOSE`
- **Flags**: specific signals with severity and evidence

In [ ]:
# Pick the first open PR dynamically
open_prs_raw = await client.list_open_prs(OWNER, REPO)
target_pr_number = open_prs_raw[0]["number"] if open_prs_raw else 1
print(f"Assessing PR #{target_pr_number}: {open_prs_raw[0]['title'] if open_prs_raw else 'N/A'}")

# Ingest and compute embedding
pr = await ingest_pr(OWNER, REPO, target_pr_number, client)
pr_embedding = compute_embedding(pr)

# Run Tier 1+2 pipeline (no LLM)
scorecard = await run_pipeline(
    pr,
    pr_embedding=pr_embedding,
    enable_tier3=False,
)

print(f"\nVerdict: {scorecard.verdict.value}")
print(f"Confidence: {scorecard.confidence:.2f}")
print(f"Summary: {scorecard.summary}")

if scorecard.flags:
    print(f"\nFlags ({len(scorecard.flags)}):")
    for flag in scorecard.flags:
        print(f"  [{flag.severity.value.upper()}] {flag.title}")
        print(f"    {flag.explanation}")
else:
    print("\nNo flags raised — clean PR.")

if scorecard.dimensions:
    print("\nDimension scores:")
    for dim in scorecard.dimensions:
        print(f"  {dim.dimension}: {dim.score:.2f} — {dim.summary}")

## 2. Batch Audit (Top 20 PRs)

Audit the 20 most recent open PRs in one call. The report includes verdict distribution,
duplicate clusters, highest-risk PRs, and contributor stats.

In [ ]:
report = await run_audit(OWNER, REPO, count=20, concurrency=3)

print(f"Audited {report.prs_analyzed} / {report.total_open_prs} open PRs in {report.elapsed_seconds:.1f}s")
print(f"\nVerdict distribution:")
print(f"  FAST_TRACK:       {report.fast_track_count}")
print(f"  REVIEW_REQUIRED:  {report.review_required_count}")
print(f"  RECOMMEND_CLOSE:  {report.recommend_close_count}")

print(f"\nContributor stats:")
print(f"  Unique authors:          {report.unique_authors}")
print(f"  First-time contributors: {report.first_time_contributors}")
print(f"  New accounts (<90d):     {report.new_accounts}")
print(f"  Sensitive path PRs:      {report.sensitive_path_prs}")
print(f"  Low test ratio PRs:      {report.low_test_prs}")

if report.clusters_090:
    print(f"\nDuplicate clusters (>=90% similarity): {len(report.clusters_090)}")
    for i, cluster in enumerate(report.clusters_090, 1):
        members = ", ".join(f"PR #{m['pr']}" for m in cluster.members)
        print(f"  Cluster {i}: {members}")

if report.highest_risk:
    print(f"\nHighest-risk PRs:")
    for entry in report.highest_risk[:5]:
        print(f"  PR #{entry.pr_number} ({entry.author}): score={entry.score:.2f}, "
              f"{entry.flag_count} flags ({entry.high_severity_count} high)")
        for flag_id in entry.flags:
            print(f"    - {flag_id}")

if report.flag_frequency:
    print(f"\nFlag frequency:")
    for flag_id, count in report.flag_frequency.items():
        print(f"  {flag_id}: {count}")

## 3. Issue-to-PR Linking

Compute embeddings for open PRs and issues, then find semantic links where contributors
didn't write `Fixes #N` explicitly.

In [ ]:
# Fetch open PRs and issues
open_prs_raw = await client.list_open_prs(OWNER, REPO)
open_issues_raw = await client.list_open_issues(OWNER, REPO)

pr_numbers = [p["number"] for p in open_prs_raw[:10]]
issue_numbers = [i["number"] for i in open_issues_raw[:10]]

print(f"Ingesting {len(pr_numbers)} PRs and {len(issue_numbers)} issues...")

prs = await ingest_batch(OWNER, REPO, pr_numbers, client)
issues = await ingest_issue_batch(OWNER, REPO, issue_numbers, client)

# Compute embeddings
pr_embeddings = [compute_embedding(pr) for pr in prs]
issue_embeddings = [compute_issue_embedding(issue) for issue in issues]

# Find links
link_report = find_issue_pr_links(prs, pr_embeddings, issues, issue_embeddings)

print(f"\nExplicit links: {len(link_report.explicit_links)}")
for link in link_report.explicit_links:
    print(f"  PR #{link.pr_number} -> Issue #{link.issue_number} (explicit)")

print(f"\nSuggested links (similarity >= {link_report.threshold}): {len(link_report.suggestions)}")
for link in link_report.suggestions[:10]:
    print(f"  PR #{link.pr_number} ({link.pr_title[:50]})")
    print(f"    -> Issue #{link.issue_number} ({link.issue_title[:50]})")
    print(f"    Similarity: {link.similarity:.3f}")

if link_report.orphan_issues:
    print(f"\nOrphan issues (no linked PRs): {link_report.orphan_issues}")

## 4. Contributor Profile

Build a profile for an active contributor — merge rate, test inclusion, areas of expertise.

In [ ]:
# Find the most active contributor from PR data
all_prs_raw = await client.list_open_prs(OWNER, REPO)
from collections import Counter
author_counts = Counter(p["user"]["login"] for p in all_prs_raw if p.get("user"))
target_user = author_counts.most_common(1)[0][0] if author_counts else "nicoseng"
print(f"Profiling contributor: {target_user}")

# Fetch their PRs (open + merged + closed)
user_prs_raw = await client.search_user_prs(OWNER, REPO, target_user, max_results=30)
user_pr_numbers = [p["number"] for p in user_prs_raw]
print(f"Found {len(user_pr_numbers)} PRs, ingesting...")

user_prs = await ingest_batch(OWNER, REPO, user_pr_numbers, client)

profile = build_contributor_profile(OWNER, REPO, target_user, user_prs)

print(f"\n{'='*50}")
print(f"Contributor: {profile.username}")
print(f"{'='*50}")
print(f"Total PRs analyzed: {profile.prs_analyzed}")
print(f"Merged: {profile.merged_prs}  |  Open: {profile.open_prs}  |  Closed: {profile.closed_prs}")
print(f"Merge rate: {profile.merge_rate:.1%}")
print(f"Test inclusion rate: {profile.test_inclusion_rate:.1%}")
print(f"Avg additions/PR: {profile.avg_additions:.0f}  |  Avg deletions/PR: {profile.avg_deletions:.0f}")
if profile.areas_of_expertise:
    print(f"Areas of expertise: {', '.join(profile.areas_of_expertise)}")
if profile.first_contribution:
    print(f"Active: {profile.first_contribution:%Y-%m-%d} → {profile.last_contribution:%Y-%m-%d}")

## 5. Cross-PR Conflict Detection

Find open PRs that modify overlapping files or similar code regions — these should be
reviewed together or sequenced to avoid merge conflicts.

In [ ]:
# Reuse or re-fetch open PRs
open_prs_raw = await client.list_open_prs(OWNER, REPO)
conflict_pr_numbers = [p["number"] for p in open_prs_raw[:20]]
print(f"Checking {len(conflict_pr_numbers)} open PRs for conflicts...")

conflict_prs = await ingest_batch(OWNER, REPO, conflict_pr_numbers, client)
conflict_embeddings = [compute_embedding(pr) for pr in conflict_prs]

conflict_report = detect_conflicts(conflict_prs, conflict_embeddings)

print(f"\nTotal open PRs checked: {conflict_report.total_open_prs}")
print(f"Conflict pairs found: {len(conflict_report.conflict_pairs)}")
print(f"Threshold: {conflict_report.threshold}  |  File overlap weight: {conflict_report.file_overlap_weight}")

if conflict_report.conflict_pairs:
    for pair in conflict_report.conflict_pairs[:10]:
        print(f"\n  PR #{pair.pr_a} vs PR #{pair.pr_b}  (confidence: {pair.confidence:.3f})")
        print(f"    A: {pair.pr_a_title[:60]}")
        print(f"    B: {pair.pr_b_title[:60]}")
        print(f"    Semantic similarity: {pair.semantic_similarity:.3f}")
        if pair.overlapping_files:
            print(f"    Overlapping files: {', '.join(pair.overlapping_files[:5])}")
else:
    print("\nNo conflicts detected among open PRs.")

## 6. Vision Document

A Vision Document defines what a project **is** and **is not** — principles, anti-patterns,
focus areas, and label taxonomy. It drives triage decisions, label automation, and staleness criteria.

Below we load the pre-generated OpenClaw vision document.

In [ ]:
import pathlib

# Look for the vision document in common locations
vision_path = None
for candidate in [
    pathlib.Path("../vision_documents/openclaw.yaml"),
    pathlib.Path("vision_documents/openclaw.yaml"),
    pathlib.Path("openclaw.yaml"),
]:
    if candidate.exists():
        vision_path = str(candidate)
        break

if vision_path:
    vision = load_vision_document(vision_path)
    print(f"Project: {vision.project}")

    print(f"\nPrinciples ({len(vision.principles)}):")
    for p in vision.principles:
        print(f"  - {p.name}")
        # Show first 120 chars of description
        desc = p.description.strip().replace('\n', ' ')[:120]
        print(f"    {desc}..." if len(p.description.strip()) > 120 else f"    {desc}")

    print(f"\nAnti-patterns ({len(vision.anti_patterns)}):")
    for ap in vision.anti_patterns:
        print(f"  - {ap}")

    print(f"\nFocus areas (sensitive paths): {vision.focus_areas}")

    if vision.label_taxonomy:
        print(f"\nLabel taxonomy ({len(vision.label_taxonomy)}):")
        for label in vision.label_taxonomy:
            print(f"  - {label.name}: {label.description}")
else:
    print("Vision document not found. Run from the repo root or adjust the path.")
    print("You can generate one with: oss-maintainer generate-vision nicoseng OpenClaw")

In [ ]:
# Cleanup
await client.__aexit__(None, None, None)
print("Done — client closed.")